<a href="https://colab.research.google.com/github/subiksha0515/Recommendation_of_RAG/blob/main/Parental_Document_Retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

🏷️ Project Title : Placement Policy Chatbot using Parent Document Retrieval (RAG)
---


---

📘 Project Overview
---

This project builds an intelligent chatbot over a college Placement Policy Handbook using a technique called Parent Document Retrieval.

Instead of answering from small text chunks (which lose rule context), the system:

Searches using small chunks for semantic accuracy

Retrieves the full original PDF page for correct rule understanding

Displays the exact page content as evidence to the user

This ensures accurate, trustworthy, and context-aware answers from large rule documents.

---

Parent Document Retrieval — Definition
--
Parent Document Retrieval is a RAG technique where small text chunks are used for semantic search, but the system retrieves and presents the full original document (parent) linked to those chunks to preserve complete context and meaning.

🎯 What This Project Helps to Understand

This project helps understand:

Why normal RAG fails on rule-based documents

The importance of context in document question answering

How to combine:

Embeddings for search

Full documents for reasoning

How real-world HR / Legal / University bots are built

How to design a two-level retrieval architecture

---

Models and Techniques Used in Each Stage
--
| Stage | What Happens Here           | Model / Tool Used                        | Technique                 | Why We Use It                              |
| ----- | --------------------------- | ---------------------------------------- | ------------------------- | ------------------------------------------ |
| 1     | Read PDF page by page       | `pypdf`                                  | Page extraction           | Treat each page as a **parent document**   |
| 2     | Split page into small parts | Python logic                             | Text chunking (300 words) | Small chunks give better embedding quality |
| 3     | Convert text to vectors     | `SentenceTransformer (all-MiniLM-L6-v2)` | Sentence embeddings       | Capture semantic meaning of chunks         |
| 4     | Store vectors for search    | `FAISS`                                  | Vector similarity search  | Quickly find relevant chunks for a query   |
| 5     | Link chunk to page          | `page_id` mapping                        | Parent ID tagging         | Connect chunk → original full page         |
| 6     | Retrieve full context       | Custom retrieval function                | Parent Document Retrieval | Show full rule, not partial text           |
| 7     | User interaction            | `Gradio`                                 | Chat UI                   | Ask questions and view source pages        |


✅ Key Idea
---
Small chunks → used for search

Full page → used for understanding the rule

This is called Parent Document Retrieval.

---

🏗️ System Architecture
---

 What Happens When User Asks a Question (Gradio UI)

Step 1 — User enters question in Gradio

Step 2 — Convert question to embedding

Step 3 — FAISS searches similar chunks

Step 4 — Retrieve the page_id of top chunks

Step 5 — Fetch FULL page text using page_id

Step 6 — Display full page content in UI


---

Cell 1 — Install libraries

In [13]:
!pip install sentence-transformers faiss-cpu pypdf gradio


Step 2 — Mount Google Drive (Parent Document Store)

Upload your PDF to Drive.

In [14]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


✅ Step 3 — Read PDF Page by Page (Parent Documents)

Each page = parent document

In [15]:
from pypdf import PdfReader

pdf_path = "/content/drive/MyDrive/CRC-Students-Placement-Handbook.pdf"
reader = PdfReader(pdf_path)

pages = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    pages.append({
        "page_id": i,
        "text": text
    })

print("Total Pages:", len(pages))


Total Pages: 25


✅ Step 4 — Create Small Chunks from Each Page (for embeddings)

In [16]:
def chunk_text(text, chunk_size=300):
    words = text.split()
    for i in range(0, len(words), chunk_size):
        yield " ".join(words[i:i+chunk_size])

chunks = []

for page in pages:
    if page["text"].strip() == "":
        continue

    for chunk in chunk_text(page["text"]):
        chunks.append({
            "chunk": chunk,
            "page_id": page["page_id"]
        })

print("Total Chunks:", len(chunks))


Total Chunks: 40


✅ Step 5 — Load Open Source Embedding Model

In [17]:
from sentence_transformers import SentenceTransformer
import numpy as np

model = SentenceTransformer('all-MiniLM-L6-v2')

chunk_texts = [c["chunk"] for c in chunks]
embeddings = model.encode(chunk_texts).astype("float32")


✅ Step 7 — Store in FAISS (Vector DB)

In [18]:
import faiss

dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print("FAISS index ready")


FAISS index ready


✅ Step 8 — Parent Document Retrieval Function (Core Part)

This is the heart.

In [19]:
def retrieve_parent_pages(query, top_k=4):
    q_emb = model.encode([query]).astype("float32")
    distances, indices = index.search(q_emb, top_k)

    parent_pages = set()
    for idx in indices[0]:
        parent_pages.add(chunks[idx]["page_id"])

    results = []
    for pid in parent_pages:
        results.append((pid, pages[pid]["text"]))

    return results


✅ Step 9 — Ask Questions (Placement Chatbot)

In [20]:
query = "Can I attend other companies after getting Dream offer?"

parent_docs = retrieve_parent_pages(query)

for i, doc in enumerate(parent_docs):
    print(f"\n--- Parent Page {i} ---\n")
    print(doc[:1500])



--- Parent Page 0 ---

(10, '1 Students should register by submitting their information in the prescribed format \nprovided by CRC.\n2 Each student can accept only one job offer. However, students who have already \nsecured a job may be allowed to participate in the selection process for their "Dream \nCompany." Once a student accepts a job offer from their dream company, they \ncannot participate in any further campus recruitment processes.\n3 Students with three or more backlogs are not allowed to register for placements. \nThey are advised to clear their backlogs before registering, unless they are extended \nstudents who have not completed their course/project requirements in their last two \nsemesters.\n4 The eligibility criteria set by visiting companies will be considered ﬁnal.\n5 Registered students must attend all training programs and workshops arranged by \nthe university or their respective departments.\n6 Students are encouraged to apply for a passport and PAN card as man

Gradio UI

In [21]:
import gradio as gr

def placement_chatbot(query):
    parents = retrieve_parent_pages(query)

    if len(parents) == 0:
        return "⚠️ No relevant content found from document."

    response = ""
    for pid, text in parents:
        response += f"\n\n🔹 **From Page {pid+1}**\n"
        response += text[:1500] + "...\n"

    return response


iface = gr.Interface(
    fn=placement_chatbot,
    inputs=gr.Textbox(label="Ask Placement Policy Question"),
    outputs=gr.Markdown(label="Relevant Rules From Handbook"),
    title="📘 Placement Policy Parent Document Retrieval Bot",
    description="Ask questions about placement rules. The bot shows exact pages from the handbook."
)

iface.launch(share=True)


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f4af7943de3354e987.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


🎯 Try These Queries

"Am I allowed to sit for another company after getting selected?"
"What happens if I skip a drive?"
"Who is final authority in placement disputes?"
"Can I contact HR directly?"


✅ Overview

This project builds a Placement Policy Chatbot using Parent Document Retrieval.
It searches the handbook using small text chunks for accurate matching, then retrieves the full original PDF page to present the complete rule with context. This ensures reliable, evidence-based answers from large policy documents.

✅ Conclusion

By combining embeddings for search and full pages for understanding, this system overcomes the common RAG problem of losing context. The result is a trustworthy, industry-style chatbot that explains rules directly from the source document.